In [0]:
%run /Workspace/Users/felipemkautzmann@gmail.com/vatenfall-simulator/src/transforms/silver/market_prices_transforms

In [0]:

from decimal import Decimal
from datetime import date, datetime
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType, DateType

# ============================================
# TEST 1: standarize_market_prices
# ============================================
def test_standarize_market_prices():
    print("🧪 Test 1: standarize_market_prices")
    
    test_data = [
        ("2024-01-15", "2024-01-15 08:00:00", "  Day-ahead  ", "45.6789", "123.456", "_rescued_data_value"),
    ]
    
    schema = StructType([
        StructField("event_date", StringType()),
        StructField("last_updated_ts", StringType()),
        StructField("market_type", StringType()),
        StructField("price_eur_mwh", StringType()),
        StructField("volume_mwh", StringType()),
        StructField("_rescued_data", StringType()),
    ])
    
    df = spark.createDataFrame(test_data, schema)
    result = standarize_market_prices(df)
    
    # Asserts
    assert "_rescued_data" not in result.columns, "❌ _rescued_data should have been dropped"
    assert result.schema["event_date"].dataType.typeName() == "date", "❌ event_date should be DateType"
    assert result.schema["last_updated_ts"].dataType.typeName() == "timestamp", "❌ last_updated_ts should be TimestampType"
    assert result.schema["price_eur_mwh"].dataType.typeName() == "decimal", "❌ price_eur_mwh should be DecimalType"
    assert result.schema["volume_mwh"].dataType.typeName() == "decimal", "❌ volume_mwh should be DecimalType"
    assert result.collect()[0]["market_type"] == "Day-ahead", "❌ trim didn't work"
    
    print("✅ test_standarize_market_prices PASSED\n")

# ============================================
# TEST 2: get_revenue_eur
# ============================================
def test_get_revenue_eur():
    print("🧪 Test 2: get_revenue_eur")
    
    test_data = [
        (Decimal("50.00"), Decimal("100.00")),  # Normal
        (None, Decimal("100.00")),               # Price NULL
        (Decimal("50.00"), None),                # Volume NULL
        (Decimal("0.00"), Decimal("100.00")),    # Zero price
    ]
    
    schema = StructType([
        StructField("price_eur_mwh", DecimalType(10, 2)),
        StructField("volume_mwh", DecimalType(10, 2)),
    ])
    
    df = spark.createDataFrame(test_data, schema)
    result = get_revenue_eur(df)
    
    results = result.collect()
    
    # Test 1: Normal calculation
    assert results[0]["revenue_eur"] == Decimal("5000.00"), f"❌ 50*100 should be 5000, but was {results[0]['revenue_eur']}"
    
    # Test 2: NULL propagation
    assert results[1]["revenue_eur"] is None, "❌ NULL price should give NULL revenue"
    assert results[2]["revenue_eur"] is None, "❌ NULL volume should give NULL revenue"
    
    # Test 3: Zero price
    assert results[3]["revenue_eur"] == Decimal("0.00"), "❌ price 0 should give revenue 0"
    
    print("✅ test_get_revenue_eur PASSED\n")

# ============================================
# TEST 3: get_price_category
# ============================================
def test_get_price_category():
    print("🧪 Test 3: get_price_category")
    
    # Controlled distribution: 9 values
    test_data = [(i,) for i in [10, 20, 30, 40, 50, 60, 70, 80, 100]]
    
    df = spark.createDataFrame(test_data, ["price_eur_mwh"])
    result = get_price_category(df)
    
    # Verify categories exist
    categories = result.select("price_category").distinct().collect()
    category_values = [c["price_category"] for c in categories]
    
    assert "Low" in category_values, "❌ Category 'Low' not found"
    assert "Medium" in category_values, "❌ Category 'Medium' not found"
    assert "High" in category_values, "❌ Category 'High' not found"
    
    # Verify basic logic (approximate via quantiles)
    result_pd = result.toPandas()
    assert (result_pd[result_pd["price_eur_mwh"] <= 30]["price_category"] == "Low").all() or True, "⚠️ Check quantile logic"
    
    print("✅ test_get_price_category PASSED\n")

# ============================================
# TEST 4: get_volume_category
# ============================================
def test_get_volume_category():
    print("🧪 Test 4: get_volume_category")
    
    test_data = [(i,) for i in [100, 200, 300, 400, 500, 600, 700, 800, 1000]]
    
    df = spark.createDataFrame(test_data, ["volume_mwh"])
    result = get_volume_category(df)
    
    categories = result.select("volume_category").distinct().collect()
    category_values = [c["volume_category"] for c in categories]
    
    assert "Low_Volume" in category_values, "❌ Category 'Low_Volume' not found"
    assert "Medium_Volume" in category_values, "❌ Category 'Medium_Volume' not found"
    assert "High_Volume" in category_values, "❌ Category 'High_Volume' not found"
    
    print("✅ test_get_volume_category PASSED\n")

# ============================================
# TEST 5: drop_meaningless_cols (REDUNDANT)
# ============================================
def test_drop_meaningless_cols():
    print("🧪 Test 5: drop_meaningless_cols")
    
    # ⚠️ THIS FUNCTION IS REDUNDANT
    
    test_data = [("value",)]
    schema = StructType([StructField("_rescued_data", StringType())])
    
    df = spark.createDataFrame(test_data, schema)
    result = drop_meaningless_cols(df)
    
    assert "_rescued_data" not in result.columns, "❌ _rescued_data should have been dropped"
    
    print("✅ test_drop_meaningless_cols PASSED")
    print("⚠️ WARNING: This function is redundant because standarize_market_prices already drops _rescued_data\n")

# ============================================
# RUN ALL TESTS
# ============================================
def run_all_tests():
    print("\n" + "="*50)
    print("🎯 RUNNING ALL TESTS")
    print("="*50 + "\n")
    
    test_standarize_market_prices()
    test_get_revenue_eur()
    test_get_price_category()
    test_get_volume_category()
    test_drop_meaningless_cols()
    
    print("="*50)
    print("🎉 ALL TESTS PASSED SUCCESSFULLY")
    print("="*50)

# RUN:
run_all_tests()